In [1]:
import pandas as pd
import numpy as np

In [35]:
# 데이터 로드
vod_meta    = pd.read_csv('../data/processed/vod_mart_processed.csv')

C:\Users\user\AppData\Local\Temp\ipykernel_15480\2338153655.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  vod_meta    = pd.read_csv('../data/processed/vod_mart_processed.csv')


In [39]:
combined = pd.read_csv('processed_data.csv')

In [40]:
combined

,asset_nm,use_tms,disp_rtm,asset_id,strt_dt_dt,user_index,genre,category_l1,category_l2,super_asset_nm,completion_rate,normalized_time
0,전국민민원해결프로젝트-일꾼의탄생 66회(23/04/12),2880,2880,M5047991LFOJ44245901,2023-05-03 22:38:06,0,시사/교양,KBS,(HD)KBS 시사교양,전국민민원해결프로젝트-일꾼의탄생,1.000000,0.033515
1,(HD)나 혼자산다 486회(23/03/17),926,5580,M5164421LFOI39723501,2023-05-03 17:24:13,1,연예/오락,MBC,(HD)MBC 연예오락,나 혼자산다,0.165950,0.010776
2,(SD)명성황후(하) 38회,500,3660,M0191290LSGK72289001,2023-05-03 16:09:53,2,미니시리즈,KBS,KBS구작,명성황후(하),0.136612,0.005819
3,꼬리에꼬리를무는그날이야기 54회(22/11/17),0,4320,M5092600LFOK59331601,2023-05-03 23:39:48,3,시사/교양,SBS,(HD)SBS 시사교양,꼬리에꼬리를무는그날이야기,0.000000,0.000000
4,골 때리는 그녀들 80회(23/02/15),5580,5580,M5066112LFOJ84983101,2023-05-03 18:48:46,4,연예/오락,SBS,(HD)SBS 연예오락,골 때리는 그녀들,1.000000,0.064935
...,...,...,...,...,...,...,...,...,...,...,...,...
17175098,헤이지니 럭키강이 시즌14 스페셜. 17회,12,1140,M4789223LSGJ69166601,2023-05-14 20:40:29,8419,기타,키즈어린이,만화동산,헤이지니 럭키강이 시즌14 스페셜.,0.010526,0.000140
17175099,한문철의 블랙박스 리뷰 12회(22/12/22),4920,4920,M5143337LFOL00172201,2023-05-14 12:17:02,8416,교양다큐,JTBC,JTBC시사교양,한문철의 블랙박스 리뷰,1.000000,0.057255
17175100,심야괴담회(정규) 14회,2769,4380,M4923883LSGJ81263001,2023-05-14 18:12:17,37101,연예/오락,MBC,MBC구작,심야괴담회(정규),0.632192,0.032223
17175101,(HD)런닝맨 601회(22/05/01),694,5220,M5042751LFOJ31748801,2023-05-14 13:29:21,67596,연예/오락,SBS,(HD)SBS 연예오락,런닝맨,0.132950,0.008076


In [42]:
combined['implicit_w']  = 1 + 4 * combined['completion_rate'] 

In [46]:
combined.dropna(inplace = True)

In [48]:
combined.isna().sum()

asset_nm           0
use_tms            0
disp_rtm           0
asset_id           0
strt_dt_dt         0
user_index         0
genre              0
category_l1        0
category_l2        0
super_asset_nm     0
completion_rate    0
normalized_time    0
implicit_w         0
dtype: int64

In [49]:
from scipy.sparse import coo_matrix
from implicit.als import AlternatingLeastSquares

# 고유 ID ↔︎ 인덱스 매핑
u_ids, u_idx = np.unique(combined['user_index'], return_inverse=True)
i_ids, i_idx = np.unique(combined['asset_id'],  return_inverse=True)

R = coo_matrix((combined['implicit_w'], (u_idx, i_idx)))        # (U×I)

als = AlternatingLeastSquares(
        factors=150, regularization=0.02, iterations=20,
        calculate_training_loss=True, dtype=np.float32)
als.fit(R)

U_latent = als.user_factors        # (U×150)
I_latent = als.item_factors        # (I×150)

c:\Users\user\anaconda3\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 1.3516745567321777 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

### 사용자 특성 (X_user)

In [53]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, hstack
from tqdm import tqdm

LONG_TERM_DAYS  = 180      # 장기 선호 집계 기간
SHORT_TERM_DAYS = 14       # 단기 선호 집계 기간

# 모든 장르 / 상위 카테고리 vocab 사전 (고정 순서 보장용)
genre_vocab = {g: i for i, g in enumerate(sorted(vod_meta['genre'].dropna().unique()))}
cat_vocab   = {c: i for i, c in enumerate(sorted(vod_meta['category_l1'].dropna().unique()))}

NUM_GENRE = len(genre_vocab)
NUM_CAT   = len(cat_vocab)

In [54]:
def pref_vector(df_hist: pd.DataFrame, col: str, vocab: dict) -> np.ndarray:
    """
    시청 히스토리 DataFrame(df_hist)에서 특정 컬럼(col)의 선호 분포(빈도 비율) 벡터를 생성.
    빈 범주가 있을 경우 0, 총합 0이면 전벡터 0.
    """
    vec = np.zeros(len(vocab), dtype=np.float32)
    counts = df_hist[col].value_counts()
    for key, cnt in counts.items():
        if key in vocab:
            vec[vocab[key]] = cnt
    total = vec.sum()
    return vec / total if total > 0 else vec

In [55]:
def build_user_profile(df_u: pd.DataFrame) -> np.ndarray:
    """
    하나의 사용자 로그(df_u)에서
    - 장기 선호 (LONG_TERM_DAYS)
    - 단기 선호 (SHORT_TERM_DAYS)
    장르 + 카테고리 벡터를 이어붙여 반환.
    shape = [2 * (NUM_GENRE + NUM_CAT), ]
    """
    now = df_u['strt_dt_dt'].max()

    # 기간 필터링
    long_hist  = df_u[df_u['strt_dt_dt'] >= now - pd.Timedelta(days=LONG_TERM_DAYS)]
    short_hist = df_u[df_u['strt_dt_dt'] >= now - pd.Timedelta(days=SHORT_TERM_DAYS)]

    # 장기
    long_genre = pref_vector(long_hist, 'genre', genre_vocab)
    long_cat   = pref_vector(long_hist, 'category_l1', cat_vocab)

    # 단기
    short_genre = pref_vector(short_hist, 'genre', genre_vocab)
    short_cat   = pref_vector(short_hist, 'category_l1', cat_vocab)

    return np.hstack([long_genre, long_cat, short_genre, short_cat]).astype(np.float32)

In [58]:
user_pref_list = []
user_ids = []

# 'strt_dt_dt' 컬럼을 datetime 타입으로 변환 (에러 방지)
combined['strt_dt_dt'] = pd.to_datetime(combined['strt_dt_dt'], errors='coerce')

for uid, df_u in tqdm(combined.groupby('user_index', sort=False)):
    user_ids.append(uid)
    user_pref_list.append(build_user_profile(df_u))

user_pref_mat = np.vstack(user_pref_list)

100%|██████████| 607713/607713 [24:31<00:00, 413.06it/s]  


In [61]:
combined['hour_angle'] = 2 * np.pi * combined['strt_dt_dt'].dt.hour     / 24.0
combined['wd_angle']   = 2 * np.pi * combined['strt_dt_dt'].dt.weekday  /  7.0

combined['hour_sin'] = np.sin(combined['hour_angle'])
combined['hour_cos'] = np.cos(combined['hour_angle'])
combined['wd_sin']   = np.sin(combined['wd_angle'])
combined['wd_cos']   = np.cos(combined['wd_angle'])

# (2) 사용자별 평균(원형 평균) 집계
time_vec_df = (combined
               .groupby('user_index')[['hour_sin','hour_cos','wd_sin','wd_cos']]
               .mean()
               .reindex(user_ids))   # 순서 맞추기

time_vec = time_vec_df.to_numpy(dtype=np.float32)

In [62]:
U_latent_aligned = U_latent

In [63]:
X_user = hstack([
            csr_matrix(user_pref_mat),           # 장/단기 선호 (dense→csr)
            csr_matrix(time_vec),                # sin / cos (dense→csr)
            csr_matrix(U_latent_aligned)         # ALS latent (dense→csr)
        ], format='csr')

print(f"[DONE] X_user shape = {X_user.shape}  (type = {type(X_user)})")

[DONE] X_user shape = (607713, 364)  (type = <class 'scipy.sparse._csr.csr_matrix'>)


### 컨텐츠 특성 (X_items)

In [64]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction import FeatureHasher  # 해싱 옵션용
from scipy.sparse import csr_matrix, hstack
from konlpy.tag import Okt
from tqdm import tqdm
import gc

#### 1. combined → 아이템별 통계(total_views·avg_watch_time) 계산

In [67]:
asset_stats = (combined
               .groupby('asset_id', as_index=False)
               .agg(total_views     = ('user_index', 'size'),
                    avg_watch_time  = ('use_tms',   'mean')))

print(f"[INFO] asset_stats rows = {len(asset_stats):,}")

[INFO] asset_stats rows = 261,664


In [70]:
asset_stats

,asset_id,total_views,avg_watch_time
0,CCS00000000000000101,3242,113.706354
1,CCS00000000000000601,1,590.000000
2,CCS00000000000001101,7,202.285714
3,CCS00000000000002801,1,81.000000
4,CCS00000000000003101,2,43.500000
...,...,...,...
261659,M5194023LSGM62415201,1,382.000000
261660,M5194024LSGM62416101,5,448.400000
261661,M5194025LSGM62417001,3,452.000000
261662,M5194026LSGM62417901,2,3187.000000


#### 2. vod 와 asset_stats 조인

In [71]:
vod_meta = vod_meta.merge(asset_stats, on='asset_id', how='left')

#### 3. 결측 보정 & dtype 캐스팅

In [80]:
vod_meta[['genre', 'category_l1', 'category_l2']].isna().sum()

genre            0
category_l1    968
category_l2    968
dtype: int64

In [82]:
for col in ['genre', 'category_l1', 'category_l2']:
    vod_meta[col] = vod_meta[col].fillna('unknown').astype(str)

for col in ['total_views', 'avg_watch_time']:
    vod_meta[col] = vod_meta[col].astype(float)

text_cols = ['asset_nm', 'smry_shrt', 'actr_disp', 'director',
             'category_l1', 'category_l2']
for col in text_cols:
    vod_meta[col] = vod_meta[col].fillna('')

In [84]:
vod_meta['genre'].nunique()

79

#### 4. 범주형 인코딩: genre → One-Hot(CSR)

In [86]:
ohe_genre = OneHotEncoder(handle_unknown='ignore', sparse_output = True)
X_genre = ohe_genre.fit_transform(vod_meta[['genre']])

#### 5. 텍스트 임베딩: (제목·줄거리·배우·감독·카테고리) → TF-IDF → SVD 200d

In [94]:
okt = Okt()

def okt_tokenizer(text: str):
    return [w for w, p in okt.pos(text, norm=True, stem=True)
            if p in ('Noun', 'Verb', 'Adjective')]
    
# --- 1) TF-IDF vectorising with progress bar ---
print("[INFO] TF-IDF vectorising ...")
tfidf = TfidfVectorizer(
    tokenizer      = okt_tokenizer,
    ngram_range    = (1, 2),
    max_features   = 20_000,
    min_df         = 5,
    dtype          = np.float32
)

# tqdm으로 감싸기만 하면, 내부에서 문서를 순회할 때마다 프로그레스 바가 갱신됩니다.
# (fit_transform은 내부적으로 fit + transform 두 번 순회하니, 두 번 바가 돕니다.)
text_corpus = vod_meta[text_cols].agg(' '.join, axis=1).tolist()
X_tfidf = tfidf.fit_transform(tqdm(text_corpus, desc="Vectorizing"))

# --- 2) TruncatedSVD 200-d with batch-wise transform ---
print("[INFO] TruncatedSVD 200-d ...")
svd = TruncatedSVD(n_components=200, random_state=42)

# 2-1) 먼저 전체 데이터로 fit
svd.fit(X_tfidf)

# 2-2) transform은 배치 단위로 쪼개서 tqdm으로 모니터링
n_samples = X_tfidf.shape[0]
batch_size = 100000   # 메모리/속도 상황에 따라 조정
svd_out = np.zeros((n_samples, svd.n_components), dtype=np.float32)

for start in tqdm(range(0, n_samples, batch_size), desc="SVD transform"):
    end = min(start + batch_size, n_samples)
    svd_out[start:end] = svd.transform(X_tfidf[start:end])

# 최종 sparse matrix가 필요하면 다시 변환
X_txt_svd = csr_matrix(svd_out)

# 메모리 해제
del X_tfidf, svd_out
gc.collect()


[INFO] TF-IDF vectorising ...


Vectorizing:   0%|          | 0/412369 [00:00<?, ?it/s]c:\Users\user\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
Vectorizing: 100%|██████████| 412369/412369 [3:59:32<00:00, 28.69it/s]  


[INFO] TruncatedSVD 200-d ...


SVD transform: 100%|██████████| 42/42 [00:01<00:00, 24.64it/s]


9

#### 6. 인기도 수치 피처

In [134]:
pop_df = vod_meta[['total_views', 'avg_watch_time']].copy()

# 0 값 많을 때는 log1p가 안정적
pop_df['total_views']    = np.log1p(pop_df['total_views'])
pop_df['avg_watch_time'] = np.log1p(pop_df['avg_watch_time'])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)    # centering 생략 → 0/0 방지
X_pop_dense = scaler.fit_transform(pop_df).astype(np.float32)

# 혹시 모를 NaN·Inf 최종 안전망
X_pop_dense = np.nan_to_num(X_pop_dense, nan=0.0, posinf=0.0, neginf=0.0)
X_pop = csr_matrix(X_pop_dense)

#### 7. hstack -> 최종 X_item

In [135]:
from scipy.sparse import hstack
X_item = hstack([X_genre, X_txt_svd, X_pop], format='csr')

### ALS - CF 후보 추출

In [105]:
import numpy as np
from scipy.sparse import csr_matrix

CF_TOP_N = 500                      # 사용자당 후보 수
csr_R    = R.tocsr()                # 반드시 CSR 형식
all_users = np.arange(R.shape[0], dtype=np.int32)    # [0, 1, …, U-1]

# ------------------------------------------------------------
# 1. ALS 벡터화 recommend (U × N 반환)
# ------------------------------------------------------------
items_mat, scores_mat = als.recommend(
    userid              = all_users,   # 전체 사용자 배열
    user_items          = csr_R,       # U × I
    N                   = CF_TOP_N,
    filter_already_liked_items = True,
    recalculate_user    = True
)

# ------------------------------------------------------------
# 2. dtype 정리 및 결측(패딩) 처리
#    implicit는 기본적으로 N개 다 채워 주지만, 필터 후 부족할 수 있어
#    -1 / -inf 가 들어오면 0 점수로 패딩
# ------------------------------------------------------------
cf_items  = items_mat.astype(np.int32)          # shape = (U, N)
cf_scores = scores_mat.astype(np.float32)       # shape = (U, N)

pad_mask  = (cf_items < 0)                      # -1 패딩 위치
cf_scores[pad_mask] = 0.0

# ------------------------------------------------------------
# 3. 완성 확인
# ------------------------------------------------------------
print(f"[DONE] cf_items shape  : {cf_items.shape}  dtype = {cf_items.dtype}")
print(f"[DONE] cf_scores shape : {cf_scores.shape} dtype = {cf_scores.dtype}")
print(f"• 패딩 비율 (items == -1) = {(pad_mask.sum()/cf_items.size):.4%}")

[DONE] cf_items shape  : (607713, 500)  dtype = int32
[DONE] cf_scores shape : (607713, 500) dtype = float32
• 패딩 비율 (items == -1) = 0.0000%


### 콘텐츠 코사인

In [139]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

X_item_norm = normalize(X_item, axis=1, copy=False)

nbrs = NearestNeighbors(
        n_neighbors=201,    # self + 200
        metric='cosine',
        algorithm='brute',
        n_jobs=-1)
nbrs.fit(X_item_norm)

dist, idx = nbrs.kneighbors(X_item_norm, return_distance=True)
sim_cb_items  = idx[:, 1:]
sim_cb_scores = 1 - dist[:, 1:]     # cosine similarity

### User - Item 특성 코사인

In [ ]:
X_user_norm = normalize(X_user, axis=1, copy=False)

def user_item_topk(U_norm, I_norm, k=500, batch=5_000):
    rows, cols, data = [], [], []
    for u0 in tqdm(range(0, U_norm.shape[0], batch), desc="UF cosine"):
        sims = U_norm[u0:u0+batch] @ I_norm.T
        sims = sims.tolil()
        for ri, vec in enumerate(sims.rows):
            vals = sims.data[ri]
            if len(vals):
                top = np.argsort(vals)[-k:]
                rows.extend([u0+ri]*len(top))
                cols.extend(np.array(vec)[top])
                data.extend(np.array(vals)[top])
    return csr_matrix((data, (rows, cols)), shape=(U_norm.shape[0], I_norm.shape[0]))

sim_uf = user_item_topk(X_user_norm, X_item_norm, k=500)


### 하이브리드 추천

In [ ]:
from sklearn.preprocessing import minmax_scale

W_CF, W_CB, W_UF = 0.60, 0.25, 0.15      # 이후 Optuna 로 튜닝

def recommend(u_idx, top_k=20):
    # -------- CF part (already min-max 0-1 스케일) --------
    items  = cf_items[u_idx]
    mask   = items >= 0
    items  = items[mask]
    s_cf   = minmax_scale(cf_scores[u_idx][mask])

    # -------- CB part : 후보 간 평균 코사인 --------
    s_cb = sim_cb[items][:, items].mean(axis=1).A1
    s_cb = minmax_scale(s_cb)

    # -------- UF part : user vs item 코사인 --------
    s_uf = sim_uf[u_idx, items].toarray().ravel()
    s_uf = minmax_scale(s_uf)

    final = W_CF*s_cf + W_CB*s_cb + W_UF*s_uf
    best  = np.argsort(-final)[:top_k]

    return i_ids[items[best]], final[best]
